# Stand up a SageMaker Studio domain with a least-privilege execution role and prove the role cannot reach buckets it does not own

The previous platform lead handed every Studio user the `AmazonSageMakerFullAccess` managed policy plus `AmazonS3FullAccess` and shipped it. Security saw an ML notebook open a compliance-team bucket last week and the lead was replaced. You inherit the Studio rollout. The CISO wants a least-privilege execution role scoped to one S3 prefix and one KMS key, a sibling-bucket deny test that survives audit, and a cleanup that does not leave the Studio backing EFS volume billing at thirty cents per GB-month for the next eighteen months. One session to ship it.

Four artifacts ship in this lab:

- A training-data S3 bucket the execution role can list and download from.
- A sibling S3 bucket the role must NOT be able to touch (deny-test fixture).
- A customer-managed KMS key whose key policy grants `kms:Decrypt` to the execution role only.
- A custom SageMaker execution role with a trust policy for the SageMaker service and an inline policy scoped to the training prefix plus the KMS key.

A Studio domain plus user profile attaches the execution role at the end so the cleanup exercise reflects the production teardown pattern. No Studio app is started; the apps are where billing happens, the domain itself is free.

**Time.** Plan for about 55 minutes hands-on. SageMaker Studio domain creation runs 3 to 5 minutes; IAM propagation adds 5 to 10 seconds on top of the AssumeRole call. Budget 90 minutes if the trust policy fights back.

**Cost.** This lab costs about a nickel if cleanup works on the first try. SageMaker Studio domain and user profile are free; only Studio apps bill, and this lab does not start one. KMS is one cent per key per month prorated to fractions of a penny per session. IAM, STS, and S3 at this scale are inside the always-free tier. The trap is the Studio domain's backing EFS volume: AWS bills it at thirty cents per gigabyte per month, forever, until you delete the domain with `HomeEfsFileSystem=Delete`. The cleanup cell sets that flag unconditionally.

This lab maps to AWS MLA-C01 Domain 1 (Data Preparation for ML, 28%) and Domain 4 (ML Solution Monitoring, Maintenance, and Security, 24%).

**SageMaker free-tier reminder.** The SageMaker free tier is 2 months from your first SageMaker resource creation, not 12 months. If this is your first SageMaker resource, the clock starts the moment `create_domain` returns.

In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT.md
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.6.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT.md
# section 12. Two bucket names use the shorter labrun-mla-lab01-* form
# because the full slug plus a 12-digit account suffix would exceed the
# 63-character S3 bucket name limit.

import atexit
import getpass
import json
import time

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)

LAB_ID = "sagemaker-studio-and-iam-execution-role"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"

# Resource names. Buckets use the abbreviated form to fit the 63-char limit.
EXEC_ROLE_NAME = f"labrun-{LAB_ID}-exec"
INLINE_POLICY_NAME = "labrun-mla-lab01-exec-policy"
KMS_ALIAS_NAME = f"alias/labrun-{LAB_ID}-key"
STUDIO_DOMAIN_NAME = f"labrun-{LAB_ID}-domain"
STUDIO_USER_NAME = "labrun-mla-lab01-user"

# Bucket names and other account-derived state are resolved after STS returns.
TRAINING_BUCKET_NAME = None
SIBLING_BUCKET_NAME = None
KMS_KEY_ID = None
STUDIO_DOMAIN_ID = None
ACCOUNT_ID = None

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# resolve account-derived names. This cell must succeed before the manifest
# cell creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"MLA Batch 1 labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")

TRAINING_BUCKET_NAME = f"labrun-mla-lab01-training-{ACCOUNT_ID}"
SIBLING_BUCKET_NAME = f"labrun-mla-lab01-sibling-{ACCOUNT_ID}"
EXEC_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{EXEC_ROLE_NAME}"
print(f"Training bucket: {TRAINING_BUCKET_NAME}")
print(f"Sibling bucket:  {SIBLING_BUCKET_NAME}")
print(f"Execution role ARN: {EXEC_ROLE_ARN}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, orphan scan.
# Manifest is module-level and in reverse-creation order. The Studio domain
# is critical because its backing EFS volume bills at $0.30/GB-month if the
# delete is called without HomeEfsFileSystem=Delete. The KMS key is critical
# because schedule_key_deletion is the only delete path and the pending
# window keeps the key alive for 7 days.
#
# The SageMaker domain and user profile do not have adapter dispatchers in
# labrun-checks 0.6.0; the cleanup cell tears them down manually BEFORE
# run_cleanup walks the manifest. The manifest below contains only
# adapter-supported types. The orphan scan still picks up any tagged
# SageMaker resources via Resource Groups Tagging API.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="iam_role",
        id=EXEC_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {EXEC_ROLE_NAME}",
    ),
    CleanupResource(
        type="kms_alias",
        id=KMS_ALIAS_NAME,
        region=REGION,
        cli_delete_command=f"aws kms delete-alias --alias-name {KMS_ALIAS_NAME}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=TRAINING_BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{TRAINING_BUCKET_NAME} --force",
    ),
    CleanupResource(
        type="s3_bucket",
        id=SIBLING_BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{SIBLING_BUCKET_NAME} --force",
    ),
]

# KMS key entry is appended once create_key returns its id. The schedule
# happens in the kms_key adapter via schedule_key_deletion(PendingWindowInDays=7).


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Run the cleanup")
    print("cell at the bottom of this notebook first, or remove them manually with")
    print("the AWS CLI commands the cleanup cell prints, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Create the training and sibling buckets, seed each with one object

The deny test in Task 4 needs two buckets: one the execution role IS allowed to read (the training bucket), and one the role is NOT allowed to touch (the sibling bucket). Both buckets need at least one object so `GetObject` has something concrete to either find or be denied on.

Build it in this order:

1. Create `labrun-mla-lab01-training-{your-account-id}` and tag it `labrun:lab-slug=sagemaker-studio-and-iam-execution-role`.
2. Create `labrun-mla-lab01-sibling-{your-account-id}` and tag it with the same lab tag (so cleanup can find it).
3. Put one tiny placeholder object under `training/seed.parquet` in the training bucket and one under `restricted/seed.parquet` in the sibling bucket. Any bytes work; the content does not matter.

Tagging at creation matters. The cleanup audit looks for the lab tag; an untagged bucket is invisible to the tag scan.

In [ ]:
# NBVAL_SKIP
# Task 1: Create both buckets, tag them, seed each with one placeholder object.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

BUCKETS = [
    (TRAINING_BUCKET_NAME, "training/seed.parquet"),
    (SIBLING_BUCKET_NAME, "restricted/seed.parquet"),
]

for bucket, key in BUCKETS:
    # YOUR CODE: Create the bucket using s3.create_bucket(Bucket=bucket).
    # us-east-1 rejects LocationConstraint in create_bucket; other regions
    # require CreateBucketConfiguration={"LocationConstraint": REGION}.

    s3.put_bucket_tagging(
        Bucket=bucket,
        Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
    )
    print(f"Bucket created and tagged: {bucket}")

    # YOUR CODE: Upload the seed object using s3.put_object() with Bucket=bucket,
    # Key=key, and Body=b"labrun seed\n".
    print(f"Uploaded s3://{bucket}/{key}")

print()
print("Both buckets are ready for the IAM scope test.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: both buckets exist with the lab tag and have at least one
# object at the expected key.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        expected = [
            (TRAINING_BUCKET_NAME, "training/seed.parquet"),
            (SIBLING_BUCKET_NAME, "restricted/seed.parquet"),
        ]
        for bucket, key in expected:
            try:
                s3_client.head_bucket(Bucket=bucket)
            except ClientError as e:
                code_str = e.response["Error"]["Code"]
                if code_str in ("404", "NoSuchBucket"):
                    return CheckpointResult(
                        status="fail",
                        error_reason=f"Bucket {bucket} does not exist.",
                    )
                return CheckpointResult(status="error", error_reason=str(e))

            try:
                tag_resp = s3_client.get_bucket_tagging(Bucket=bucket)
                tags = {t["Key"]: t["Value"] for t in tag_resp.get("TagSet", [])}
            except ClientError as e:
                if e.response["Error"]["Code"] == "NoSuchTagSet":
                    tags = {}
                else:
                    return CheckpointResult(status="error", error_reason=str(e))
            if tags.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Bucket {bucket} is missing tag {LAB_TAG_KEY}={LAB_TAG_VALUE}. "
                        f"Found tags: {tags}"
                    ),
                )

            try:
                s3_client.head_object(Bucket=bucket, Key=key)
            except ClientError as e:
                code_str = e.response["Error"]["Code"]
                if code_str in ("404", "NoSuchKey"):
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"Bucket {bucket} has no object at {key}. Upload one before "
                            f"the IAM scope test can run."
                        ),
                    )
                return CheckpointResult(status="error", error_reason=str(e))

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

Two buckets, two seed objects. The checkpoint tells you whether a bucket is missing, untagged, or empty at the expected key. Read the failure message before peeking further.

</details>

<details><summary>Hint 2 (direction)</summary>

In `us-east-1` use plain `s3.create_bucket(Bucket=bucket)` with no `CreateBucketConfiguration`. The `put_bucket_tagging` call is already wired in the cell; you only need to fill in the two missing lines. `s3.put_object(Bucket=bucket, Key=key, Body=b"labrun seed\n")` is the simplest upload.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

for bucket, key in [(TRAINING_BUCKET_NAME, "training/seed.parquet"),
                    (SIBLING_BUCKET_NAME, "restricted/seed.parquet")]:
    s3.create_bucket(Bucket=bucket)
    s3.put_bucket_tagging(
        Bucket=bucket,
        Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
    )
    s3.put_object(Bucket=bucket, Key=key, Body=b"labrun seed\n")
```

</details>

**Wallet check.** Two empty buckets, two tiny seed objects. S3 control-plane plus four object operations sit comfortably inside the always-free tier. Damage so far: about $0.00. Your coffee this morning cost infinitely more.

## Task 2: Create a customer-managed KMS key and an alias, then attach a key policy that lets the execution role Decrypt

The execution role's inline policy in Task 3 will grant `kms:Decrypt` on a specific key ARN. AWS requires the key policy to ALSO grant that permission to the role; KMS is one of the few AWS services where the resource policy is mandatory, not optional. Without the key policy grant, the role's IAM policy alone is not enough.

Build it in this order:

1. Call `kms.create_key` with a description and the lab tag, and capture the returned `KeyId`. The cell appends a KMS-key entry to `CLEANUP_MANIFEST` immediately because schedule_key_deletion is the only way to remove it (a 7-day pending window applies).
2. Create the alias `alias/labrun-sagemaker-studio-and-iam-execution-role-key` pointing at the key. Aliases are how human-readable names map to KMS key IDs.
3. Attach a key policy: one statement granting `kms:*` to the account root (mandatory to keep the key administrable), one statement granting `kms:Decrypt` to the execution role ARN you will create in Task 3.

The execution role ARN is already constructed in the setup cell as `EXEC_ROLE_ARN`. KMS does not require the principal to exist yet at the time `put_key_policy` runs; the policy can reference a future role ARN. The role is created in Task 3.

In [ ]:
# NBVAL_SKIP
# Task 2: Create the KMS key, alias, and key policy.

kms = boto3.client(
    "kms",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: Create the KMS key using kms.create_key() with
# Description="labrun mla lab 01 customer-managed key" and
# Tags=[{"TagKey": LAB_TAG_KEY, "TagValue": LAB_TAG_VALUE}]. Assign the
# response to create_resp.

KMS_KEY_ID = create_resp["KeyMetadata"]["KeyId"]
KMS_KEY_ARN = create_resp["KeyMetadata"]["Arn"]
print(f"Created KMS key: {KMS_KEY_ID}")

# Register the key for cleanup immediately. schedule_key_deletion is the only
# delete path; the 7-day pending window is the AWS minimum.
CLEANUP_MANIFEST.append(
    CleanupResource(
        type="kms_key",
        id=KMS_KEY_ID,
        region=REGION,
        cli_delete_command=(
            f"aws kms schedule-key-deletion --key-id {KMS_KEY_ID} --pending-window-in-days 7"
        ),
    )
)

try:
    # YOUR CODE: Create the alias using kms.create_alias() with
    # AliasName=KMS_ALIAS_NAME and TargetKeyId=KMS_KEY_ID.
    print(f"Created alias: {KMS_ALIAS_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] != "AlreadyExistsException":
        raise
    print(f"Alias {KMS_ALIAS_NAME} already exists, reusing.")

key_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "EnableIAMUserPermissions",
            "Effect": "Allow",
            "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:root"},
            "Action": "kms:*",
            "Resource": "*",
        },
        {
            "Sid": "AllowExecRoleDecrypt",
            "Effect": "Allow",
            "Principal": {"AWS": EXEC_ROLE_ARN},
            "Action": "kms:Decrypt",
            "Resource": "*",
        },
    ],
}

# YOUR CODE: Attach the key policy using kms.put_key_policy() with
# KeyId=KMS_KEY_ID, PolicyName="default", and Policy=json.dumps(key_policy).

print("Key policy attached.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: KMS key exists with the lab tag, alias resolves to the key,
# and the key policy grants kms:Decrypt to the execution role ARN.

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        kms_client = boto3.client(
            "kms",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            aliases = kms_client.list_aliases()
        except ClientError as e:
            return CheckpointResult(status="error", error_reason=str(e))
        target_alias = next(
            (a for a in aliases.get("Aliases", []) if a.get("AliasName") == KMS_ALIAS_NAME),
            None,
        )
        if target_alias is None:
            return CheckpointResult(
                status="fail",
                error_reason=f"Alias {KMS_ALIAS_NAME} not found.",
            )
        target_key_id = target_alias.get("TargetKeyId")
        if not target_key_id:
            return CheckpointResult(
                status="fail",
                error_reason=f"Alias {KMS_ALIAS_NAME} is not pointing at a key.",
            )

        try:
            tags_resp = kms_client.list_resource_tags(KeyId=target_key_id)
            tag_map = {t["TagKey"]: t["TagValue"] for t in tags_resp.get("Tags", [])}
        except ClientError as e:
            return CheckpointResult(status="error", error_reason=str(e))
        if tag_map.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"KMS key {target_key_id} is missing tag {LAB_TAG_KEY}={LAB_TAG_VALUE}. "
                    f"Found: {tag_map}"
                ),
            )

        try:
            policy_resp = kms_client.get_key_policy(KeyId=target_key_id, PolicyName="default")
        except ClientError as e:
            return CheckpointResult(status="error", error_reason=str(e))
        policy_doc = json.loads(policy_resp["Policy"])
        statements = policy_doc.get("Statement", [])

        decrypt_grants = {"kms:Decrypt", "kms:*", "*"}
        granted = False
        for stmt in statements:
            if stmt.get("Effect") != "Allow":
                continue
            principal = stmt.get("Principal", {})
            aws_principal = principal.get("AWS") if isinstance(principal, dict) else None
            principals = (
                [aws_principal] if isinstance(aws_principal, str)
                else list(aws_principal) if isinstance(aws_principal, list)
                else []
            )
            if EXEC_ROLE_ARN not in principals:
                continue
            actions = stmt.get("Action", [])
            if isinstance(actions, str):
                actions = [actions]
            if any(a in decrypt_grants for a in actions):
                granted = True
                break
        if not granted:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Key policy does not grant kms:Decrypt to Principal {EXEC_ROLE_ARN!r}. "
                    f"Wildcard kms:* or * also accepted."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

Three calls: `create_key`, `create_alias`, `put_key_policy`. The checkpoint tells you whether the alias is missing, the tag is missing, or the key policy does not grant `kms:Decrypt` to the execution role ARN.

</details>

<details><summary>Hint 2 (direction)</summary>

`kms.create_key` takes Tags in the form `[{"TagKey": ..., "TagValue": ...}]`, not the EC2-style `[{"Key": ..., "Value": ...}]`. Capture `response["KeyMetadata"]["KeyId"]`. The key policy `Statement` list needs the account-root `kms:*` grant first (omit it and you lock yourself out of the key administration) and the role-ARN `kms:Decrypt` grant second.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
kms = boto3.client(
    "kms",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

create_resp = kms.create_key(
    Description="labrun mla lab 01 customer-managed key",
    Tags=[{"TagKey": LAB_TAG_KEY, "TagValue": LAB_TAG_VALUE}],
)
KMS_KEY_ID = create_resp["KeyMetadata"]["KeyId"]
KMS_KEY_ARN = create_resp["KeyMetadata"]["Arn"]

kms.create_alias(AliasName=KMS_ALIAS_NAME, TargetKeyId=KMS_KEY_ID)

key_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {"Sid": "EnableIAMUserPermissions",
         "Effect": "Allow",
         "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:root"},
         "Action": "kms:*", "Resource": "*"},
        {"Sid": "AllowExecRoleDecrypt",
         "Effect": "Allow",
         "Principal": {"AWS": EXEC_ROLE_ARN},
         "Action": "kms:Decrypt", "Resource": "*"},
    ],
}
kms.put_key_policy(KeyId=KMS_KEY_ID, PolicyName="default", Policy=json.dumps(key_policy))
```

</details>

**Wallet check.** KMS bills one cent per key per month, prorated to fractions of a penny for a 90-minute session. The first customer-managed key in a new account is free for the first month. Damage so far: still effectively $0.00.

## Task 3: Create the SageMaker execution role with a trust policy and a prefix-scoped inline policy

This is where the exam-relevant cognitive work lives. Three pieces have to line up:

1. A trust policy with `Principal.Service: sagemaker.amazonaws.com` so the SageMaker service can assume the role. Without this the Studio user profile cannot adopt it.
2. An inline policy with three statements: one allowing `s3:GetObject` and `s3:PutObject` on `arn:aws:s3:::{training-bucket}/training/*`, one allowing `s3:ListBucket` on the training bucket ARN with a `Condition: StringLike: s3:prefix: training/*` (the prefix Condition is what scopes ListBucket; without it the role can enumerate the entire bucket), and one allowing `kms:Decrypt` on the lab KMS key ARN.
3. The role must NOT grant any permission on the sibling bucket. The deny test in Task 4 will prove this.

Why three statements instead of one combined statement? `s3:GetObject` and `s3:PutObject` take object-key ARNs; `s3:ListBucket` takes the bucket ARN itself (no key suffix). They cannot share a `Resource` line. Mixing them up is the single most common scope bug on SAA-style and MLA-style exam questions.

In [ ]:
# NBVAL_SKIP
# Task 3: Create the execution role and attach the prefix-scoped inline policy.

iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "sagemaker.amazonaws.com",
                "AWS": f"arn:aws:iam::{ACCOUNT_ID}:root",
            },
            "Action": "sts:AssumeRole",
        }
    ],
}

try:
    # YOUR CODE: Create the role using iam.create_role() with
    # RoleName=EXEC_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy),
    # Description="labrun mla lab 01 execution role", and
    # Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
    print(f"Created role: {EXEC_ROLE_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        print(f"Role {EXEC_ROLE_NAME} already exists, reusing.")
    else:
        raise

inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "TrainingPrefixRW",
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:PutObject"],
            "Resource": f"arn:aws:s3:::{TRAINING_BUCKET_NAME}/training/*",
        },
        {
            "Sid": "TrainingBucketList",
            "Effect": "Allow",
            "Action": "s3:ListBucket",
            "Resource": f"arn:aws:s3:::{TRAINING_BUCKET_NAME}",
            "Condition": {
                "StringLike": {"s3:prefix": ["training/*"]}
            },
        },
        {
            "Sid": "LabKeyDecrypt",
            "Effect": "Allow",
            "Action": "kms:Decrypt",
            "Resource": KMS_KEY_ARN,
        },
    ],
}

# YOUR CODE: Attach the inline policy using iam.put_role_policy() with
# RoleName=EXEC_ROLE_NAME, PolicyName=INLINE_POLICY_NAME, and
# PolicyDocument=json.dumps(inline_policy).

print(f"Attached inline policy {INLINE_POLICY_NAME} to {EXEC_ROLE_NAME}")

# IAM propagation. The AssumeRole call in Task 4 is stricter than most IAM
# calls; give the new role 10 seconds to land.
print("Your IAM role is stuck in traffic, give it 10 seconds...")
time.sleep(10)
print("Done.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: execution role exists with the SageMaker trust policy and
# an inline policy that scopes S3 + KMS to the training prefix and key.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        iam_client = boto3.client(
            "iam",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            role_resp = iam_client.get_role(RoleName=EXEC_ROLE_NAME)
        except ClientError as e:
            if e.response["Error"]["Code"] == "NoSuchEntity":
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Role {EXEC_ROLE_NAME} does not exist.",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        trust = role_resp["Role"]["AssumeRolePolicyDocument"]
        services = []
        for stmt in trust.get("Statement", []):
            p = stmt.get("Principal", {})
            svc = p.get("Service")
            if isinstance(svc, str):
                services.append(svc)
            elif isinstance(svc, list):
                services.extend(svc)
        if "sagemaker.amazonaws.com" not in services:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Trust policy on {EXEC_ROLE_NAME} does not include "
                    f"Service principal sagemaker.amazonaws.com. Found services: {services}."
                ),
            )

        inline = iam_client.list_role_policies(RoleName=EXEC_ROLE_NAME)
        names = inline.get("PolicyNames", [])
        if not names:
            return CheckpointResult(
                status="fail",
                error_reason=f"Role {EXEC_ROLE_NAME} has no inline policy attached.",
            )
        doc_resp = iam_client.get_role_policy(RoleName=EXEC_ROLE_NAME, PolicyName=names[0])
        doc = doc_resp["PolicyDocument"]

        action_wildcards = {"s3:*", "*"}
        kms_wildcards = {"kms:*", "*"}
        required_s3_rw = {"s3:GetObject", "s3:PutObject"}
        list_bucket_arn = f"arn:aws:s3:::{TRAINING_BUCKET_NAME}"
        training_obj_arn = f"arn:aws:s3:::{TRAINING_BUCKET_NAME}/training/*"

        # Confirm rw on training prefix.
        rw_ok = False
        list_ok = False
        kms_ok = False
        for stmt in doc.get("Statement", []):
            if stmt.get("Effect") != "Allow":
                continue
            actions = stmt.get("Action", [])
            if isinstance(actions, str):
                actions = [actions]
            action_set = set(actions)
            resources = stmt.get("Resource", [])
            if isinstance(resources, str):
                resources = [resources]
            resource_set = set(resources)

            covers_rw = required_s3_rw.issubset(action_set) or bool(action_set & action_wildcards)
            if covers_rw and training_obj_arn in resource_set:
                rw_ok = True

            covers_list = "s3:ListBucket" in action_set or bool(action_set & action_wildcards)
            if covers_list and list_bucket_arn in resource_set:
                cond = stmt.get("Condition", {}) or {}
                prefix_values = []
                for op in ("StringLike", "StringEquals"):
                    vals = cond.get(op, {}).get("s3:prefix", [])
                    if isinstance(vals, str):
                        prefix_values.append(vals)
                    elif isinstance(vals, list):
                        prefix_values.extend(vals)
                if any("training/" in v for v in prefix_values):
                    list_ok = True

            covers_decrypt = "kms:Decrypt" in action_set or bool(action_set & kms_wildcards)
            if covers_decrypt:
                # Accept the lab key ARN, the lab key id, or a wildcard.
                if any(
                    KMS_KEY_ID in r or r == "*" or r.endswith(KMS_KEY_ID)
                    for r in resource_set
                ):
                    kms_ok = True

        if not rw_ok:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Inline policy is missing an Allow on s3:GetObject/s3:PutObject scoped to "
                    f"{training_obj_arn!r}."
                ),
            )
        if not list_ok:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Inline policy is missing s3:ListBucket on {list_bucket_arn!r} with a "
                    f"Condition: StringLike: s3:prefix containing 'training/'. The prefix "
                    f"Condition is what scopes the list to the right folder."
                ),
            )
        if not kms_ok:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Inline policy is missing kms:Decrypt scoped to the lab KMS key ARN "
                    f"(key id {KMS_KEY_ID})."
                ),
            )

        # Negative assertion: no Resource that covers the sibling bucket.
        sibling_root = f"arn:aws:s3:::{SIBLING_BUCKET_NAME}"
        for stmt in doc.get("Statement", []):
            resources = stmt.get("Resource", [])
            if isinstance(resources, str):
                resources = [resources]
            for r in resources:
                if r == "*" or r == "arn:aws:s3:::*" or sibling_root in r:
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"Inline policy includes a Resource {r!r} that covers the sibling "
                            f"bucket. The role must NOT have any S3 permission on "
                            f"{SIBLING_BUCKET_NAME}."
                        ),
                    )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint reads the trust policy first, then walks every statement of the inline policy looking for three things: object-level S3 RW on the training prefix, bucket-level ListBucket with the right prefix Condition, and KMS Decrypt on the lab key. It also asserts the policy does NOT cover the sibling bucket. The error message tells you which of those is wrong.

</details>

<details><summary>Hint 2 (direction)</summary>

`s3:GetObject` and `s3:PutObject` need a Resource ARN with `/training/*` on the end. `s3:ListBucket` needs the bare bucket ARN (`arn:aws:s3:::bucket-name`) and a `Condition: StringLike: s3:prefix: ["training/*"]` block. The Condition is the part that scopes list to one folder. `kms:Decrypt` needs the key ARN (not the alias name). The trust policy's `Principal.Service` is `sagemaker.amazonaws.com`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow",
         "Principal": {"Service": "sagemaker.amazonaws.com",
                       "AWS": f"arn:aws:iam::{ACCOUNT_ID}:root"},
         "Action": "sts:AssumeRole"}
    ],
}
iam.create_role(
    RoleName=EXEC_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(trust_policy),
    Description="labrun mla lab 01 execution role",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)

inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow",
         "Action": ["s3:GetObject", "s3:PutObject"],
         "Resource": f"arn:aws:s3:::{TRAINING_BUCKET_NAME}/training/*"},
        {"Effect": "Allow",
         "Action": "s3:ListBucket",
         "Resource": f"arn:aws:s3:::{TRAINING_BUCKET_NAME}",
         "Condition": {"StringLike": {"s3:prefix": ["training/*"]}}},
        {"Effect": "Allow",
         "Action": "kms:Decrypt",
         "Resource": KMS_KEY_ARN},
    ],
}
iam.put_role_policy(
    RoleName=EXEC_ROLE_NAME,
    PolicyName=INLINE_POLICY_NAME,
    PolicyDocument=json.dumps(inline_policy),
)
```

</details>

**Wallet check.** IAM `CreateRole`, `PutRolePolicy`, and a 10-second sleep do not move the AWS bill. Damage so far: still effectively $0.00. The compliance team is the only thing accruing cost right now (their time, waiting on the audit response).

## Task 4: Stand up the Studio domain, attach the user profile, and prove the AssumeRole deny test

Two things have to be true at the end of this task. First, the SageMaker Studio domain exists with the user profile attached to the execution role (this is the SageMaker-side binding the brief calls for). Second, the IAM evaluation rejects the role on the sibling bucket: an `sts.assume_role` into the execution role yields temporary credentials that succeed on `training/seed.parquet` and fail with `AccessDenied` on `restricted/seed.parquet`.

Build it in this order:

1. Find the default VPC and the first subnet in the account. Studio domain creation requires VPC and subnet IDs even with `AppNetworkAccessType=PublicInternetOnly`.
2. Call `sagemaker.create_domain` with `AuthMode=IAM`, `DefaultUserSettings.ExecutionRoleArn=EXEC_ROLE_ARN`, `AppNetworkAccessType=PublicInternetOnly` (avoids needing a NAT Gateway), and the lab tag. Domain creation runs 3 to 5 minutes; the cell waits for `Status=InService`.
3. Call `sagemaker.create_user_profile` for the single user. The user profile inherits the execution role from the domain default.
4. Call `sts.assume_role` into the execution role. Build a fresh boto3 S3 client from the returned temporary credentials (including `aws_session_token`). Reusing your base credentials tests your own permissions, not the role.
5. Call `get_object` on `training/seed.parquet` (expect 200) and on `restricted/seed.parquet` (expect `ClientError` with code `AccessDenied`).

The deny test is the proof. Inline policy plus trust policy plus a clean negative result is what the exam tests.

In [ ]:
# NBVAL_SKIP
# Task 4: Studio domain, user profile, AssumeRole deny test.

sagemaker = boto3.client(
    "sagemaker",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
ec2 = boto3.client(
    "ec2",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
sts = boto3.client(
    "sts",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Resolve the default VPC and one subnet.
vpcs = ec2.describe_vpcs(Filters=[{"Name": "isDefault", "Values": ["true"]}])["Vpcs"]
if not vpcs:
    print("No default VPC found in this account. Studio domain requires a VPC id.")
    raise SystemExit(1)
VPC_ID = vpcs[0]["VpcId"]
subnet_resp = ec2.describe_subnets(Filters=[{"Name": "vpc-id", "Values": [VPC_ID]}])
SUBNET_IDS = [s["SubnetId"] for s in subnet_resp["Subnets"][:1]]
print(f"Default VPC: {VPC_ID}, subnet: {SUBNET_IDS[0]}")

# Create the Studio domain. PublicInternetOnly avoids needing a NAT Gateway.
create_domain_resp = sagemaker.create_domain(
    DomainName=STUDIO_DOMAIN_NAME,
    AuthMode="IAM",
    DefaultUserSettings={"ExecutionRoleArn": EXEC_ROLE_ARN},
    SubnetIds=SUBNET_IDS,
    VpcId=VPC_ID,
    AppNetworkAccessType="PublicInternetOnly",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
STUDIO_DOMAIN_ID = create_domain_resp["DomainArn"].split("/")[-1]
print(f"Created Studio domain: {STUDIO_DOMAIN_ID}. Waiting for InService...")
print("Teaching the Studio domain how to make coffee. This takes 3 to 5 minutes...")

elapsed = 0
while elapsed < 600:
    desc = sagemaker.describe_domain(DomainId=STUDIO_DOMAIN_ID)
    status = desc["Status"]
    if status == "InService":
        print(f"Studio domain ready: {STUDIO_DOMAIN_ID}")
        break
    if status in ("Failed", "Delete_Failed"):
        print(f"Studio domain entered {status}. Stop and investigate.")
        raise SystemExit(1)
    time.sleep(15)
    elapsed += 15
else:
    print("Studio domain did not reach InService within 10 minutes.")
    raise SystemExit(1)

# Create the user profile.
sagemaker.create_user_profile(
    DomainId=STUDIO_DOMAIN_ID,
    UserProfileName=STUDIO_USER_NAME,
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
print(f"Created Studio user profile: {STUDIO_USER_NAME}")

# AssumeRole deny test.
# YOUR CODE: Call sts.assume_role(RoleArn=EXEC_ROLE_ARN, RoleSessionName="labrun-deny-test")
# and store the response in assume_resp.

temp_creds = assume_resp["Credentials"]

# YOUR CODE: Build a NEW boto3 S3 client from temp_creds. Pass
# aws_access_key_id=temp_creds["AccessKeyId"],
# aws_secret_access_key=temp_creds["SecretAccessKey"],
# aws_session_token=temp_creds["SessionToken"], region_name=REGION.
# Assign it to s3_assumed.

# Training read should succeed.
resp = s3_assumed.get_object(Bucket=TRAINING_BUCKET_NAME, Key="training/seed.parquet")
print(f"Training read OK: training/seed.parquet returned {resp['ResponseMetadata']['HTTPStatusCode']}")

# Sibling read should fail with AccessDenied.
try:
    s3_assumed.get_object(Bucket=SIBLING_BUCKET_NAME, Key="restricted/seed.parquet")
    print("UNEXPECTED PASS: sibling read should have been denied")
except ClientError as e:
    code_str = e.response["Error"]["Code"]
    print(f"Sibling read denied as expected: {code_str}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: AssumeRole into the execution role yields temp credentials
# that succeed on training/seed.parquet and fail with AccessDenied on
# restricted/seed.parquet. Wraps the calls in an exponential-backoff retry
# capped at 60 seconds for IAM eventual consistency.

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        sts_client = boto3.client(
            "sts",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        last_error = None
        delay = 2
        elapsed = 0
        assume_resp = None
        while elapsed < 60:
            try:
                assume_resp = sts_client.assume_role(
                    RoleArn=EXEC_ROLE_ARN,
                    RoleSessionName="labrun-checkpoint-4",
                )
                last_error = None
                break
            except ClientError as e:
                last_error = e
                time.sleep(delay)
                elapsed += delay
                delay = min(delay * 2, 8)
        if last_error is not None or assume_resp is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Could not assume {EXEC_ROLE_ARN} within 60s. STS returned "
                    f"{last_error.response['Error']['Code']}: "
                    f"{last_error.response['Error']['Message']}. Check the trust policy."
                ),
            )

        temp_creds = assume_resp["Credentials"]
        s3_assumed = boto3.client(
            "s3",
            aws_access_key_id=temp_creds["AccessKeyId"],
            aws_secret_access_key=temp_creds["SecretAccessKey"],
            aws_session_token=temp_creds["SessionToken"],
            region_name=REGION,
        )

        try:
            s3_assumed.get_object(Bucket=TRAINING_BUCKET_NAME, Key="training/seed.parquet")
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Execution role could not read training/seed.parquet. Got "
                    f"{e.response['Error']['Code']}. Verify the inline policy Resource for "
                    f"s3:GetObject covers arn:aws:s3:::{TRAINING_BUCKET_NAME}/training/*."
                ),
            )

        try:
            s3_assumed.get_object(Bucket=SIBLING_BUCKET_NAME, Key="restricted/seed.parquet")
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Execution role successfully read the sibling bucket. The inline "
                    f"policy is over-broad. Check that no statement has Resource = '*' or "
                    f"any ARN covering {SIBLING_BUCKET_NAME}."
                ),
            )
        except ClientError as e:
            code_str = e.response["Error"]["Code"]
            if code_str != "AccessDenied":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Expected AccessDenied on the sibling read, got {code_str}."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

Two failure shapes are possible. Either the role cannot read its own prefix (something denied a path that should be allowed) or the role CAN read the sibling bucket (the inline policy is over-broad). The checkpoint message tells you which one.

</details>

<details><summary>Hint 2 (direction)</summary>

The most common miss is forgetting to build a NEW boto3 client from `temp_creds`. Reusing your base `s3` client tests your own permissions, not the role. Pass `aws_session_token=temp_creds["SessionToken"]` to the new client. If assume_role itself fails, the trust policy `Principal` does not match your caller account, or IAM has not propagated yet; the validator retries for 60 seconds.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for the Task 4 deny test:

```python
assume_resp = sts.assume_role(
    RoleArn=EXEC_ROLE_ARN,
    RoleSessionName="labrun-deny-test",
)
temp_creds = assume_resp["Credentials"]

s3_assumed = boto3.client(
    "s3",
    aws_access_key_id=temp_creds["AccessKeyId"],
    aws_secret_access_key=temp_creds["SecretAccessKey"],
    aws_session_token=temp_creds["SessionToken"],
    region_name=REGION,
)

s3_assumed.get_object(Bucket=TRAINING_BUCKET_NAME, Key="training/seed.parquet")
try:
    s3_assumed.get_object(Bucket=SIBLING_BUCKET_NAME, Key="restricted/seed.parquet")
except ClientError as e:
    assert e.response["Error"]["Code"] == "AccessDenied"
```

</details>

**Wallet check.** SageMaker Studio domain creation and the user profile are free; only Studio apps bill, and this lab does not start one. STS AssumeRole, three GetObject calls. Damage so far: about $0.01. The KMS key is the only thing still accruing cost, and at prorated $0.001 per session you would need 50 of these labs back to back to outspend your morning coffee.

## Cleanup

The Studio domain teardown is the longest single step here. `delete_domain` must be called with `RetentionPolicy.HomeEfsFileSystem=Delete` or the backing EFS volume keeps billing at $0.30/GB-month indefinitely. The user profile inside the domain must be deleted first. The cell below does both manually (labrun-checks 0.6.0 does not have a SageMaker adapter yet), then runs the standard three-phase cleanup against the IAM, KMS, and S3 resources.

Re-running the cell is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Studio user profile, then Studio domain with EFS delete, then
# run_cleanup on the remaining adapter-supported resources.

import sys

sagemaker_cleanup = boto3.client(
    "sagemaker",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

manual_warnings: list[str] = []

if STUDIO_DOMAIN_ID:
    # Delete the user profile first; delete_domain rejects domains with live profiles.
    try:
        sagemaker_cleanup.delete_user_profile(
            DomainId=STUDIO_DOMAIN_ID,
            UserProfileName=STUDIO_USER_NAME,
        )
        print(f"Deleted user profile {STUDIO_USER_NAME}")
        # Wait for the profile to disappear.
        elapsed = 0
        while elapsed < 180:
            try:
                sagemaker_cleanup.describe_user_profile(
                    DomainId=STUDIO_DOMAIN_ID,
                    UserProfileName=STUDIO_USER_NAME,
                )
                time.sleep(10)
                elapsed += 10
            except ClientError as e:
                if e.response["Error"]["Code"] in ("ResourceNotFound", "ValidationException"):
                    break
                raise
    except ClientError as e:
        if e.response["Error"]["Code"] not in ("ResourceNotFound", "ValidationException"):
            manual_warnings.append(
                f"FAILED TO DELETE: user_profile {STUDIO_USER_NAME!r}. Error: {e}. "
                f"Run manually: aws sagemaker delete-user-profile --domain-id "
                f"{STUDIO_DOMAIN_ID} --user-profile-name {STUDIO_USER_NAME}"
            )

    try:
        sagemaker_cleanup.delete_domain(
            DomainId=STUDIO_DOMAIN_ID,
            RetentionPolicy={"HomeEfsFileSystem": "Delete"},
        )
        print(f"Requested delete on domain {STUDIO_DOMAIN_ID} with HomeEfsFileSystem=Delete")
        # Wait for the domain to disappear so the EFS teardown completes.
        elapsed = 0
        while elapsed < 300:
            try:
                sagemaker_cleanup.describe_domain(DomainId=STUDIO_DOMAIN_ID)
                time.sleep(15)
                elapsed += 15
            except ClientError as e:
                if e.response["Error"]["Code"] in ("ResourceNotFound", "ValidationException"):
                    print("Studio domain is gone and the backing EFS volume was torn down.")
                    break
                raise
    except ClientError as e:
        if e.response["Error"]["Code"] not in ("ResourceNotFound", "ValidationException"):
            manual_warnings.append(
                f"FAILED TO DELETE: sagemaker_domain {STUDIO_DOMAIN_ID!r}. Error: {e}. "
                f"Run manually: aws sagemaker delete-domain --domain-id {STUDIO_DOMAIN_ID} "
                f"--retention-policy HomeEfsFileSystem=Delete"
            )

result = run_cleanup(CLEANUP_MANIFEST)

for warning in manual_warnings:
    print(warning)
for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if manual_warnings or result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

manual_total = 2  # user profile + domain
manual_failed = len(manual_warnings)
critical_total = 2  # studio domain + kms key
critical_destroyed = critical_total - (
    (1 if any("sagemaker_domain" in w for w in manual_warnings) else 0)
    + (1 if KMS_KEY_ID and KMS_KEY_ID in failed_ids else 0)
)
standard_total = len(CLEANUP_MANIFEST) - 1 + manual_total - 2  # exclude kms key, add manual user profile, exclude domain since both counted as critical
# Simpler accounting: report standard as everything in manifest minus kms key, plus user profile.
standard_total = (len(CLEANUP_MANIFEST) - 1) + 1
standard_destroyed = standard_total - (len(failed_ids) - (1 if KMS_KEY_ID in failed_ids else 0)) - (
    1 if any("user_profile" in w for w in manual_warnings) else 0
)
failed_count = len(failed_ids) + len(manual_warnings)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")
if KMS_KEY_ID and KMS_KEY_ID not in failed_ids:
    print(
        f"Note: KMS key {KMS_KEY_ID} entered the 7-day pending-deletion window. "
        f"It will be permanently deleted by AWS after that period. This is expected."
    )

final_status = "clean" if (failed_count == 0 and result.status == "clean") else "dirty"
cleanup(status=final_status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $0.01.** Studio domain creation and the user profile are free; only Studio apps bill, and this lab did not start one. KMS was a fraction of a penny prorated. IAM, STS, and two tiny S3 buckets sat inside the always-free tier. The backing EFS volume was torn down by the `HomeEfsFileSystem=Delete` flag on `delete_domain`. Check your AWS Billing console in 24 hours to confirm zero ongoing charges.

## Reflection

These are not graded. They are for you.

1. Walk through what AWS evaluates when a Studio user runs a notebook cell that calls `boto3.client("s3").get_object(...)`. Trace the IAM evaluation path: the kernel's runtime credentials come from the execution role, the request hits S3 with those credentials, S3 evaluates the IAM policy attached to the role, the bucket policy on the target bucket, and (if applicable) the KMS key policy if the object is SSE-KMS-encrypted. Which of these would have caught the cross-bucket access in the scenario, and which would have missed it?

2. Why is `AmazonSageMakerFullAccess` the wrong managed policy to attach as a Studio execution role in production? Walk through the specific permissions it grants that would expose a cross-team data leak (consider S3, EC2, KMS, and IAM PassRole). What is the minimum set of permissions you would attach instead for a team whose notebooks only need to read one training-data bucket?

## Exam-Style Practice

**Question 1 (Domain 4, SageMaker execution role least privilege):**

A data team is provisioning a SageMaker Studio domain for the analytics group. The platform team wants to enforce least privilege: notebooks should be able to read the `s3://analytics-training/` bucket and decrypt objects encrypted with the `analytics-cmk` KMS key, and nothing else. Which configuration meets the requirement with the least operational overhead?

A. Attach the `AmazonSageMakerFullAccess` managed policy to the Studio user profile's execution role and rely on the bucket policy on `analytics-training` to deny everything else.

B. Create a custom execution role with a SageMaker service trust policy and one inline policy granting `s3:GetObject`, `s3:PutObject`, and `s3:ListBucket` (with `Condition: StringLike: s3:prefix: */*`) on the `analytics-training` bucket and `kms:Decrypt` on `analytics-cmk`.

C. Attach `AmazonS3ReadOnlyAccess` and `AWSKeyManagementServicePowerUser` to the execution role, scoped via SCPs at the organization level.

D. Use the default Studio execution role and add a bucket policy on every other bucket in the account explicitly denying the role's ARN.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: `AmazonSageMakerFullAccess` grants `s3:GetObject` on `*` and a broad set of EC2, IAM PassRole, ECR, and KMS permissions. Layering a bucket policy on a single bucket does not contain a role that has `s3:*` on every other bucket in the account.
- B is correct: a custom trust policy with `Principal.Service: sagemaker.amazonaws.com` plus a scoped inline policy is the AWS-recommended least-privilege pattern for SageMaker execution roles. The `s3:prefix` Condition on ListBucket scopes the role to the target prefix and the KMS Decrypt permission is scoped to a single key.
- C is wrong: `AmazonS3ReadOnlyAccess` grants `s3:Get*` on every bucket in every region. `AWSKeyManagementServicePowerUser` grants Decrypt on every key. Scoping with SCPs is operationally heavier and does not get you below the union of those two managed policies.
- D is wrong: it inverts the IAM model. The right pattern is to grant only what the role needs, not to deny everything the role does not need (the account has potentially hundreds of buckets, many of them not yet created).

</details>

**Question 2 (Domain 4, SageMaker Studio domain cleanup cost trap):**

A team deleted a SageMaker Studio domain via the AWS Console six months ago. Their AWS bill still shows $40 per month under "Amazon EFS." Investigation reveals no EFS file systems are visible in the EFS Console. What happened, and how should the team prevent this in the future?

A. The Studio domain was deleted but its backing EFS volume was preserved because `RetentionPolicy.HomeEfsFileSystem` defaults to `Retain`. The team should locate the orphan EFS volume in the EFS API (not the Console) and delete it. To prevent recurrence, always call `delete_domain` with `RetentionPolicy.HomeEfsFileSystem=Delete`.

B. EFS bills for 30 days after the last access regardless of deletion. The team's bill will drop to zero next billing cycle automatically.

C. The Studio domain's EFS volume is actually deleted; the bill is for CloudWatch Logs retention from past Studio sessions.

D. The team is being charged for SageMaker Notebook Instances, not EFS. The EFS line item is a billing console mislabeling.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: SageMaker Studio creates a backing EFS file system at domain creation time. `delete_domain` defaults to `RetentionPolicy.HomeEfsFileSystem=Retain` to protect notebook state. The EFS volume persists and continues to bill $0.30/GB-month indefinitely. It is queryable via `aws efs describe-file-systems` but does not surface in the Studio Console. The fix is to pass `--retention-policy HomeEfsFileSystem=Delete` on the domain delete.
- B is wrong: EFS bills storage indefinitely until the volume is deleted; there is no grace period.
- C is wrong: CloudWatch Logs retention is billed under CloudWatch, not EFS; the line item would be labeled `AmazonCloudWatchLogs`.
- D is wrong: SageMaker Notebook Instance billing appears under `AmazonSageMaker`, not `Amazon EFS`. The mislabeling theory is wrong; the EFS volume is real.

</details>